<a href="https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/01-agentic-rag/hw-01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Homework: Agentic RAG]('https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/01-agentic-rag/hw-01.ipynb')

In this homework, we build a RAG system from scratch and then make it
agentic - the same path as the module.

Instead of the course FAQ, our knowledge base is the course lessons
themselves.

The course repository is organized by module. Each module is a top-level
folder with a `lessons/` subfolder of numbered markdown pages:

```
01-agentic-rag/
└── lessons/
    ├── 01-intro.md
    ├── 02-environment.md
    ├── ...
    └── 16-other-frameworks.md
```

There are seven modules:

- `01-agentic-rag`
- `02-vector-search`
- `03-orchestration`
- `04-evaluation`
- `05-monitoring`
- `06-best-practices`
- `07-project-example`

Each lesson page is a single markdown file. These pages are exactly what you
read as you go through the course.

We'll fetch this data from GitHub and use it as the knowledge base for our
RAG system.

> It's possible your answers won't match exactly. If so, select the closest one.

## Setup

Prepare your environment the same way as in the module's
[Environment](../../../01-agentic-rag/lessons/02-environment.md) lesson.

This homework needs one extra library: `gitsource`, which downloads files
from a GitHub repository.

Install it:

```bash
uv add gitsource
```

For the LLM, we recommend OpenAI with `llama-3.3-70b-versatile`, but you can use any model
and provider you like - just adapt the client and the usage fields accordingly.

## Preparation

First, we will pull the lesson pages straight from the course repository.
We will use the commit `8c1834d` to make sure everyone works with the exact same data.

We will use `gitsource` for that:

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
```

`GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. Because we specify `allowed_extensions={"md"}`, it only checks  the markdown files.

We also pass a `filename_filter` so we don't grab every markdown file in the
repo, like the top-level README. The lesson pages all live under a module's `lessons/` folder, so
filtering on `/lessons/` keeps just those.


Each file has a `parse()` method that returns a dictionary with its
`filename` and `content`:

```python
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
```

In [18]:
# Install necessary Python packages for data processing, search, LLM interaction, environment variable management, and Git data reading.
%pip install requests minsearch groq gitsource sqlitesearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.1 MB/s eta 0:00:00


## Q1. How many lesson pages

How many lesson pages are in the dataset?

In [19]:
from gitsource import GithubRepositoryDataReader

# Initialize GithubRepositoryDataReader to read markdown files from a specific GitHub repository commit.
# It filters for files in the 'lessons/' path.
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [20]:
# Read the files matching the criteria.
documents = [file.parse() for file in reader.read()]

# Display the first document to inspect its structure and content.
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [21]:
# Print the total number of lesson pages (documents) found.
print(f"Q1: {len(documents)}")

Q1: 72


## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

In [22]:
import minsearch

# Initialize minsearch.Index, specifying 'content' as a text field for full-text search
# and 'filename' as a keyword field for exact matching.
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
# Fit the index with the loaded documents.
index.fit(documents)

In [23]:
# Define the search query for Q2.
query = "How does the agentic loop keep calling the model until it stops?"

In [24]:
# Perform a search on the index with the defined query and retrieve only the top 1 result.
result_search = index.search(query, num_results=1)

In [25]:
# Print the filename of the first result for Q2, as determined by manual inspection or prior execution.
print(f"length context: {len(result_search[0]['content'])} simbols")
print(f"Q2: {result_search[0]['filename']}")

length context: 10121 simbols
Q2: 01-agentic-rag/lessons/14-agentic-loop.md


In [26]:
import time
from sqlitesearch import TextSearchIndex

index_sql = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lesson_llm_course.db"
)

for doc in documents:
    index_sql.add(doc)
    print(f"""Added: {doc["filename"]}...""")
    time.sleep(0.5)

index_sql.close()
print("Done. Index saved to lesson_llm_course.db")

Added: 01-agentic-rag/lessons/01-intro.md...
Added: 01-agentic-rag/lessons/02-environment.md...
Added: 01-agentic-rag/lessons/03-rag.md...
Added: 01-agentic-rag/lessons/04-dataset.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/06-building-prompt.md...
Added: 01-agentic-rag/lessons/07-llm.md...
Added: 01-agentic-rag/lessons/08-rag-helper.md...
Added: 01-agentic-rag/lessons/09-data-ingestion.md...
Added: 01-agentic-rag/lessons/10-rag-next-steps.md...
Added: 01-agentic-rag/lessons/11-agents-intro.md...
Added: 01-agentic-rag/lessons/12-rag-revision.md...
Added: 01-agentic-rag/lessons/13-function-calling.md...
Added: 01-agentic-rag/lessons/14-agentic-loop.md...
Added: 01-agentic-rag/lessons/15-frameworks.md...
Added: 01-agentic-rag/lessons/16-other-frameworks.md...
Added: 02-vector-search/lessons/01-intro.md...
Added: 02-vector-search/lessons/02-embeddings.md...
Added: 02-vector-search/lessons/03-embeddings-dataset.md...
Added: 02-vector-search/lessons/04-

In [27]:
index_sql.count()

72

In [30]:
import os

file_path = 'lesson_llm_course.db'
file_size_mb = os.path.getsize(file_path) / 1024 ** 2


print(f"File size of {file_path}: {file_size_mb:.2f} MB)")

File size of lesson_llm_course.db: 1.02 MB)


In [31]:
results = index_sql.search(query, num_results=72)
sql_result_search = [doc["filename"] for doc in results]

In [32]:
print(f"Count files: {len(sql_result_search)}")
print(f"Q2: {sql_result_search[0]}")

Count files: 59
Q2: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use `llama-3.3-70b-versatile`. How many input (prompt) tokens did we send to the model for
this request?

In [33]:
import os
from groq import Groq
from google.colab import userdata

api_key_groq = userdata.get('GROQ_API_KEY')

# Initialize the Groq client with the retrieved API key.
client_groq = Groq(api_key=api_key_groq)

In [ ]:
# Send a chat completion request to the Groq API using the 'llama-3.1-8b-instant' model.
chat_completion = client_groq.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.1-8b-instant",
)

# Print the content of the assistant's response.
print(chat_completion.choices[0].message.content)

In [34]:
# Define the system instructions for the RAG assistant.
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

# Define the prompt template that structures the question and context for the LLM.
PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [36]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-24 00:27:13--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-24 00:27:13 (26.7 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [38]:
from dataclasses import dataclass
from rag_helper import RAGBase as OriginalRAGBase

# Define a dataclass to store the result of a RAG query, including the answer and token counts.
@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    output_tokens: int

# Define the RAGBase class by inheriting from OriginalRAGBase and adapting it for our schema.
class RAGBase(OriginalRAGBase):

    # self,
    # index,
    # llm_client,
    # instructions=INSTRUCTIONS,
    # prompt_template=PROMPT_TEMPLATE,

    # Method to perform a search using the provided index.
    # Adapted for the filename/content schema.
    def search(self, query, num_results=1):
        return self.index.search(query, num_results=num_results)

    # Method to build context string from search results.
    # Adapted for the filename/content schema.
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    # Method to interact with the LLM.
    # Overridden to use chat.completions.create and return the full response object for token counting.
    def llm(self, prompt):
        input_messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        return response

    # Main RAG method to perform the entire RAG process.
    # Overridden to return a RAGResult with token counts.
    def rag(self, query):
        search_results = self.search(query)
        # print(search_results)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)


        # Extract the answer and token usage from the LLM response.
        answer = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        return RAGResult(answer=answer, input_tokens=input_tokens, output_tokens=output_tokens)

In [39]:
# Initialize RAGBase with the document index, Groq client, and specified model.
rag = RAGBase(
    index=index_sql,
    llm_client=client_groq,
    model="llama-3.3-70b-versatile"
)
# Run the RAG query.
result = rag.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q3.
Q3_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q3_input_tokens)

The agentic loop keeps calling the model until it stops by using a `while` loop that continues to iterate as long as the model returns a response with a function call. The loop checks for the presence of a function call in the model's response using the `has_function_calls` flag. If the response contains a function call, the flag is set to `True`, and the loop continues. If the response does not contain a function call, the flag is set to `False`, and the loop exits.

Here is the relevant code snippet:
```python
while True:
    ...
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = Tr

In [40]:
print(rag.build_prompt)

<bound method RAGBase.build_prompt of <__main__.RAGBase object at 0x799c7776c980>>


In [41]:
# Print the input and output tokens for Q3.
print(f"""Q3:
input tokens: {Q3_input_tokens}
output tokens: {result.output_tokens}""")

Q3:
input tokens: 2399
output tokens: 297


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

In [42]:
from gitsource import chunk_documents

# Chunk the documents into smaller pieces with a specified size and step for overlapping.
chunks = chunk_documents(documents, size=2000, step=1000)

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

In [43]:
# Print the total number of chunks created.
print(f"Q4: {len(chunks)}")

Q4: 295


## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

# алгоритм дій:
1. Розбиваємо документи на chunks.
2. Індексуємо chunks замість цілих документів.
3. Пошук повертає лише ті chunks, де є релевантний текст.
4. Формуємо prompt із цих chunks.

In [ ]:
import minsearch
# Initialize a new minsearch.Index for the chunks, with 'content' as text and 'filename' as keyword.
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
# Fit the index with the generated chunks.
chunk_index.fit(chunks)

In [44]:
import time
from sqlitesearch import TextSearchIndex

# Initialize a new TextSearchIndex for chunks
chunk_index_sql = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lesson_llm_course_chunks.db" # Using a new database file for chunks
)

# Add each chunk to the new SQL index
for i, chunk in enumerate(chunks):
    chunk_index_sql.add(chunk)
    # print(f"Added chunk {i+1} from {chunk['filename']}...")
    # time.sleep(0.01) # Small delay to avoid overwhelming the system if many chunks

chunk_index_sql.close()
print("Done. Chunk index saved to lesson_llm_course_chunks.db")

Done. Chunk index saved to lesson_llm_course_chunks.db


In [45]:
import os

file_path_chunks = 'lesson_llm_course_chunks.db'
file_size_mb_chunks = os.path.getsize(file_path_chunks)/ 1024 ** 2

print(f"File size of {file_path_chunks}: {file_size_mb_chunks:.2f} MB)")

File size of lesson_llm_course_chunks.db: 1.94 MB)


In [47]:
# Initialize RAGBase with the chunked index, Groq client, and specified model.
rag_helper = RAGBase(
    index=chunk_index_sql,
    llm_client=client_groq,
    model="llama-3.3-70b-versatile",
    # model="openai/gpt-oss-120b"
)
# Run the RAG query with the chunked data.
result = rag_helper.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q5.
Q5_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q5_input_tokens)
# Print the output tokens used.
print("Output tokens:", result.output_tokens)

The agentic loop keeps calling the model until it stops by using a `while True` loop. The loop continues to execute indefinitely until the exit condition is met. The exit condition is that the model returns a response without any function calls, i.e., `has_function_calls` is `False`. 

In the code, this is achieved by the following lines:

```python
while True:
    ...
    if has_function_calls == False:
        break
```

This means that as long as the model returns a response with function calls (`has_function_calls` is `True`), the loop will continue to call the model. Once the model returns a response without any function calls (`has_function_calls` is `False`), the loop will exit the `while True` loop using the `break` statement.
Input tokens: 575
Output tokens: 165


In [48]:
# Calculate and print how many times fewer input tokens were used in Q5 compared to Q3.
print(f"Q5: {int(Q3_input_tokens/Q5_input_tokens)}x fewer")

Q5: 4x fewer


## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

In [54]:
import json

# Initialize a counter for search calls.
search_call_count = 0

# Define the search function, which simulates searching the course lessons.
# It increments a global counter and uses chunk_index to perform the actual search.
def search(query: str):
    """Search the course lessons for relevant content."""
    global search_call_count
    search_call_count += 1

    results = chunk_index_sql.search(query, num_results=5)
    return results

# Define available functions mapping their names to the actual function objects.
available_functions = {"search": search}

# Helper function to execute a function call (currently only 'search').
# It parses arguments, calls the function, and formats the output.
def make_call(tool_call, available_functions):
    """
    Выполняет функцию, запрошенную моделью, и возвращает сообщение
    в формате, который понимает Groq API.

    :param tool_call: Объект ChatCompletionMessageToolCall из ответа LLM
    :param available_functions: Словарь вида {'имя_функции': объект_функции_python}
    """
    function_name = tool_call.function.name
    # Аргументы приходят в виде строки JSON, их нужно распарсить
    function_args = json.loads(tool_call.function.arguments)

    print(f"Выполняю функцию: {function_name} с аргументами {function_args}")

    # Поиск функции в словаре доступных инструментов
    function_to_call = available_functions.get(function_name)

    if function_to_call:
        try:
            # Вызываем функцию
            function_response = function_to_call(**function_args)
        except Exception as e:
            function_response = f"Ошибка при выполнении функции: {str(e)}"
    else:
        function_response = f"Ошибка: Функция {function_name} не найдена."

    # Groq API требует, чтобы результат был строкой
    content = json.dumps(function_response, ensure_ascii=False)

    # Возвращаем словарь в формате сообщения 'tool'
    return {
        "role": "tool",
        "tool_call_id": tool_call.id, # ID, который прислала LLM для этого вызова
        "name": function_name,
        "content": content
    }

# Define the schema for the 'search' tool, describing its type, name, description, and parameters.
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course lessons knowledge base for relevant content.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query string to find relevant course lessons."
                }
            },
            "required": ["query"]
        }
    }
}


# Define the instructions for the developer role of the agent.
instructions = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""

# Define the user's question for the agent.
question = "How does the agentic loop work, and how is it different from plain RAG?"

# Initialize the list of messages, starting with system instructions and the user's question.
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

# Reset search call count and iteration counter.
search_call_count = 0
it = 1
last_answer = ""

# Initialize token usage counters.
total_input_tokens = 0
total_output_tokens = 0

# Start the agentic loop.
while True:
    print(f"iteration #{it}...")

    # Send messages to the LLM with the search tool available.
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=[search_tool],
        tool_choice="auto"
    )

    # Accumulate token usage from the LLM response.
    total_input_tokens += response.usage.prompt_tokens
    total_output_tokens += response.usage.completion_tokens

    # Get the LLM's message (assistant's response)
    llm_message = response.choices[0].message

    # Manually construct the dictionary for the assistant's message
    # to ensure only supported fields are included for the next API call.
    assistant_message_dict = {
        "role": llm_message.role,
        "content": llm_message.content,
    }
    if llm_message.tool_calls:
        # Convert Pydantic tool_calls objects to dictionaries for clean append
        # using model_dump(exclude_unset=True) to only include explicitly set fields.
        assistant_message_dict["tool_calls"] = [
            tool_call.model_dump(exclude_unset=True) for tool_call in llm_message.tool_calls
        ]
    messages.append(assistant_message_dict)

    has_function_calls = False # Reset for this iteration

    # Check if the LLM's message includes tool calls
    if llm_message.tool_calls:
        has_function_calls = True
        print(f"LLM requested tool calls: {len(llm_message.tool_calls)}")
        for tool_call in llm_message.tool_calls:
            # Execute the tool call
            tool_message = make_call(tool_call, available_functions)
            # Add the tool's output to the conversation history
            messages.append(tool_message)
    elif llm_message.content:
        # If no tool calls, but content is present, this is the final answer
        last_answer = llm_message.content
        print("ASSISTANT:")
        print(last_answer)
        # No function calls, so has_function_calls remains False
    else:
        print("No tool calls and no content in LLM message.")

    print(messages) # Debugging output

    # Increment iteration counter.
    it = it + 1
    # Break the loop if no function calls were made in this iteration,
    # meaning the LLM provided a final textual answer or no action.
    if not has_function_calls:
        print("No function calls detected or final answer provided, breaking loop.")
        break

print(f"\n--- Number of Search Calls ---")
print(f"The agent called search {search_call_count} times")

# Example pricing for Groq's llama-3.3-70b-versatile (check Groq's official pricing for actual values)
# Assuming: Input token price = $0.70 per 1M tokens, Output token price = $0.80 per 1M tokens
price_per_million_input_tokens = 0.70
price_per_million_output_tokens = 0.80

# Calculate the estimated total cost based on token usage and example pricing.
total_cost = (
    (total_input_tokens / 1_000_000) * price_per_million_input_tokens +
    (total_output_tokens / 1_000_000) * price_per_million_output_tokens
)

print(f"\n--- Token Usage and Cost ---")
print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")
print(f"Estimated Total Cost: ${total_cost:.4f} (based on example pricing)")

iteration #1...
LLM requested tool calls: 3
Выполняю функцию: search с аргументами {'query': 'agentic loop vs plain RAG'}
Выполняю функцию: search с аргументами {'query': 'agentic loop model explanation'}
Выполняю функцию: search с аргументами {'query': 'key differences between agentic loop and RAG'}
[{'role': 'system', 'content': "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."}, {'role': 'user', 'content': 'How does the agentic loop work, and how is it different from plain RAG?'}, {'role': 'assistant', 'content': None, 'tool_calls': [{'id': '6pwqa7gbg', 'function': {'arguments': '{"query":"agentic loop vs plain RAG"}', 'name': 'search'}, 'type': 'function'}, {'id': '7rs6ve0xq', 'function': {'arguments': '{"query":"agentic loop model explanation"}', 'name': 'search'}, 'type': 'function'}, {'id': 'w1msj2r1s', 'function': {'arguments': '{"query":"key differences between agentic loo

The agentic loop is a framework that allows the LLM to decide which actions to take and in which order, whereas plain RAG is a fixed flow that doesn't allow for dynamic decision-making. The agentic loop involves a while loop that calls the LLM, executes any tool calls it returns, sends the results back, and stops when the model produces a final answer with no more tool calls. This pattern is the foundation of every agent framework. In Kestra, the AIAgent plugin handles that loop for you. You define the goal, the tools, and optionally a system message - Kestra drives the loop, manages conversation history, and surfaces the result as a task output.
[{'role': 'system', 'content': "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."}, {'role': 'user', 'content': 'How does the agentic loop work, and how is it different from plain RAG?'}, {'role': 'assistant', 'content': None, 'tool_calls': [{'id': '6pwqa7gbg', 'function': {'arguments': '{"query":"agentic loop vs plain RAG"}', 'name': 'search'}, 'type': 'function'}, {'id': '7rs6ve0xq', 'function': {'arguments': '{"query":"agentic loop model explanation"}', 'name': 'search'}, 'type': 'function'}, {'id': 'w1msj2r1s', 'function': {'arguments': '{"query":"key differences between agentic loop and RAG"}', 'name': 'search'}, 'type': 'function'}]}, {'role': 'tool', 'tool_call_id': '6pwqa7gbg', 'name': 'search', 'content': '[{"start": 5000, "content": "unction, and serialize the result.\\n\\n```python\\nimport json\\n\\ncall = response.output[0]\\nargs = json.loads(call.arguments)\\n\\nresults = search(**args)\\nresult_json = json.dumps(results, indent=2)\\n```\\n\\nNow we send this result back to the model. First, we add the model\'s\\noutput to the conversation history - the model needs to see its own\\nfunction call. Then we add the tool result.\\n\\n```python\\nmessages.extend(response.output)\\n\\nmessages.append({\\n    \\"type\\": \\"function_call_output\\",\\n    \\"call_id\\": call.call_id,\\n    \\"output\\": result_json,\\n})\\n```\\n\\nThe `call_id` links the tool result to the specific function call the\\nmodel requested. If the model makes multiple function calls in one\\nturn, each one gets its own `call_id`.\\n\\n## Asking the model again\\n\\nWe call the API a second time with the expanded history:\\n\\n```python\\nresponse = openai_client.responses.create(\\n    model=\\"gpt-5.4-mini\\",\\n    input=messages,\\n    tools=[search_tool],\\n)\\n\\nresponse.output_text\\n```\\n\\nThis time the model has the original question, its own decision to\\ncall `search`, and the FAQ results. It can now produce a proper\\ncourse-specific answer.\\n\\nWe have to send the whole history because LLMs are stateless between\\nAPI calls. The memory is the list you send as `input`. If you send\\nonly the tool result, the model has no idea what\'s going on. So on\\nthis second call we replay everything we have so far. That means the\\nquestion, the decision to call `search`, and the result we got back.\\n\\nThat\'s the full function-calling loop for a single turn. With plain\\nRAG we made one call, and here we make two. Turning RAG agentic means\\nmore round-trips.\\n\\nPeople call this pattern \\"agentic RAG\\", \\"tool use\\", or \\"function\\ncalling\\". The idea behind all of them is the same. The LLM decides\\nwhich tools to call.\\n\\n## Token usage and cost\\n\\nWe just made two API calls instead of one. Each call we send to the\\nmodel costs money, so it\'s worth checking how much one tool-using turn\\nactually costs.\\n\\nThe response has a `usage` field with the token counts:\\n\\n", "filename": "01-agentic-rag/lessons/13-function-calling.md"}, {"start": 6000, "content": "its own decision to\\ncall `search`, and the FAQ results. It can now produce a proper\\ncourse-specific answer.\\n\\nWe have to send the whole history because LLMs are stateless between\\nAPI calls. The memory is the list you send as `input`. If you send\\nonly the tool result, the model has no idea what\'s going on. So on\\nthis second call we replay everything we have so far. That means the\\nquestion, the decision to call `search`, and the result we got back.\\n\\nThat\'s the full function-calling loop for a single turn. With plain\\nRAG we made one call, and here we make two. Turning RAG agentic means\\nmore round-trips.\\n\\nPeople call this pattern \\"agentic RAG\\", \\"tool use\\", or \\"function\\ncalling\\". The idea behind all of them is the same. The LLM decides\\nwhich tools to call.\\n\\n## Token usage and cost\\n\\nWe just made two API calls instead of one. Each call we send to the\\nmodel costs money, so it\'s worth checking how much one tool-using turn\\nactually costs.\\n\\nThe response has a `usage` field with the token counts:\\n\\n```python\\nusage = response.usage\\nusage.input_tokens, usage.output_tokens\\n```\\n\\nFor each model the provider publishes a price per million input tokens\\nand per million output tokens. Plug those numbers in to convert tokens\\nto dollars.\\n\\n```python\\ndef calculate_gpt54mini_price(input_tokens, output_tokens):\\n    INPUT_PRICE_PER_MILLION = 0.15\\n    OUTPUT_PRICE_PER_MILLION = 0.60\\n\\n    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION\\n    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION\\n    total_cost = input_cost + output_cost\\n\\n    return {\\n        \\"input_cost\\": input_cost,\\n        \\"output_cost\\": output_cost,\\n        \\"total_cost\\": total_cost,\\n    }\\n\\nresult = calculate_gpt54mini_price(652, 33)\\nprint(\\"Total cost: $\\", round(result[\\"total_cost\\"], 8))\\n```\\n\\nThis usage is only for the second API call. The first call also has\\nits own usage and its own cost. That was the call where the model\\ndecided to invoke `search`. Two calls means we pay twice. We pay even\\nmore on", "filename": "01-agentic-rag/lessons/13-function-calling.md"}, {"start": 4000, "content": "esult.\\n\\nThe `result` is a `LoopResult` with `all_messages` (the full\\nconversation), token counts, and `cost` (computed from token usage).\\n\\n## Cost and tokens\\n\\nLook at what the call cost:\\n\\n```python\\nresult.cost\\n```\\n\\nUseful while developing - especially with multi-turn agents where one\\nprompt can trigger several model calls. The handwritten loop made you\\ncompute this by hand. The framework keeps a running total for you.\\n\\nYou can also look at the full message history.\\n\\n```python\\nresult.all_messages\\n```\\n\\nThis is just a list - the same `messages` list we maintained by hand.\\n\\n## Continuing the conversation\\n\\nTake the messages from the previous result and pass them as\\n`previous_messages` on the next `loop` call:\\n\\n```python\\nresult2 = runner.loop(\\n    prompt=\\"How do I run a different model?\\",\\n    previous_messages=result.all_messages,\\n    callback=callback,\\n)\\n```\\n\\nThe runner picks up where the last call left off, with the same agent\\nloop and an extended history. The model knows \\"different model\\" refers\\nto Ollama because it sees the previous turn in memory. Without that\\nhistory, it would have no idea what we\'re asking about.\\n\\n## Interactive chat\\n\\nFor a chat-like workflow, run the built-in input loop:\\n\\n```python\\nrunner.run()\\n```\\n\\nType questions and get answers. Type \\"stop\\" to exit.\\n\\n[← The Agentic Loop](14-agentic-loop.md) | [Other Frameworks →](16-other-frameworks.md)", "filename": "01-agentic-rag/lessons/15-frameworks.md"}, {"start": 0, "content": "# AI Agents\\n\\nVideo: [Agentic Workflows](https://youtu.be/7tvpR8EE0gs)\\n\\nIn [Module 1](../../01-agentic-rag/lessons/11-agents-intro.md) you built the agentic loop by hand: a `while` loop that called the LLM, executed any tool calls it returned, sent the results back, and stopped when the model produced a final answer with no more tool calls. That pattern is the foundation of every agent framework.\\n\\nIn Kestra, the `AIAgent` plugin handles that loop for you. You define the goal, the tools, and optionally a system message - Kestra drives the loop, manages conversation history, and surfaces the result as a task output.\\n\\n> Note: The flows in this lesson use `{{ secret(\'GEMINI_API_KEY\') }}`. Make sure you\'ve completed the [setup instructions](03-setup.md) to configure this secret before running them.\\n\\nThe example flows use Gemini, but the `provider` block supports any major AI provider — swap `io.kestra.plugin.ai.provider.GoogleGemini` for `OpenAI`, `Anthropic`, or others. See the [full list of supported providers](https://kestra.io/plugins/plugin-ai/provider).\\n\\nTraditional Workflow — fixed sequence, predetermined logic:\\n\\n```yaml\\ntasks:\\n  - id: step1\\n    type: Task1\\n  - id: step2\\n    type: Task2\\n  - id: step3\\n    type: Task3\\n```\\n\\nAI Agent Workflow — agent decides what to do, in what order, based on the goal:\\n\\n```yaml\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.agent.AIAgent\\n    prompt: \\"Research data engineering trends and create a report\\"\\n    tools:\\n      - WebSearch\\n      - TaskExecution\\n```\\n\\n## When to Use AI Agents\\n\\nUse AI Agents when the exact sequence of steps isn\'t known in advance, decisions depend on dynamic changing information, or you need to adapt to unexpected conditions.\\n\\nUse traditional workflows when steps are deterministic and repeatable, compliance requires exact auditable processes, or cost and latency must be minimized.\\n\\n## Anatomy of an AI Agent\\n\\n```yaml\\nid: example_agent\\nnamespace: zoomcamp\\n\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.age", "filename": "03-orchestration/lessons/06-agents.md"}, {"start": 7000, "content": "```python\\nusage = response.usage\\nusage.input_tokens, usage.output_tokens\\n```\\n\\nFor each model the provider publishes a price per million input tokens\\nand per million output tokens. Plug those numbers in to convert tokens\\nto dollars.\\n\\n```python\\ndef calculate_gpt54mini_price(input_tokens, output_tokens):\\n    INPUT_PRICE_PER_MILLION = 0.15\\n    OUTPUT_PRICE_PER_MILLION = 0.60\\n\\n    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION\\n    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION\\n    total_cost = input_cost + output_cost\\n\\n    return {\\n        \\"input_cost\\": input_cost,\\n        \\"output_cost\\": output_cost,\\n        \\"total_cost\\": total_cost,\\n    }\\n\\nresult = calculate_gpt54mini_price(652, 33)\\nprint(\\"Total cost: $\\", round(result[\\"total_cost\\"], 8))\\n```\\n\\nThis usage is only for the second API call. The first call also has\\nits own usage and its own cost. That was the call where the model\\ndecided to invoke `search`. Two calls means we pay twice. We pay even\\nmore on the second call, because we resend the full history as input.\\n\\nWith a real agent loop the model can make many calls, so the costs add\\nup. Keep an eye on `usage` while you develop.\\n\\n[← Quick RAG Revision](12-rag-revision.md) | [The Agentic Loop →](14-agentic-loop.md)", "filename": "01-agentic-rag/lessons/13-function-calling.md"}]'}, {'role': 'tool', 'tool_call_id': '7rs6ve0xq', 'name': 'search', 'content': '[{"start": 4000, "content": "esult.\\n\\nThe `result` is a `LoopResult` with `all_messages` (the full\\nconversation), token counts, and `cost` (computed from token usage).\\n\\n## Cost and tokens\\n\\nLook at what the call cost:\\n\\n```python\\nresult.cost\\n```\\n\\nUseful while developing - especially with multi-turn agents where one\\nprompt can trigger several model calls. The handwritten loop made you\\ncompute this by hand. The framework keeps a running total for you.\\n\\nYou can also look at the full message history.\\n\\n```python\\nresult.all_messages\\n```\\n\\nThis is just a list - the same `messages` list we maintained by hand.\\n\\n## Continuing the conversation\\n\\nTake the messages from the previous result and pass them as\\n`previous_messages` on the next `loop` call:\\n\\n```python\\nresult2 = runner.loop(\\n    prompt=\\"How do I run a different model?\\",\\n    previous_messages=result.all_messages,\\n    callback=callback,\\n)\\n```\\n\\nThe runner picks up where the last call left off, with the same agent\\nloop and an extended history. The model knows \\"different model\\" refers\\nto Ollama because it sees the previous turn in memory. Without that\\nhistory, it would have no idea what we\'re asking about.\\n\\n## Interactive chat\\n\\nFor a chat-like workflow, run the built-in input loop:\\n\\n```python\\nrunner.run()\\n```\\n\\nType questions and get answers. Type \\"stop\\" to exit.\\n\\n[← The Agentic Loop](14-agentic-loop.md) | [Other Frameworks →](16-other-frameworks.md)", "filename": "01-agentic-rag/lessons/15-frameworks.md"}, {"start": 0, "content": "# AI Agents\\n\\nVideo: [Agentic Workflows](https://youtu.be/7tvpR8EE0gs)\\n\\nIn [Module 1](../../01-agentic-rag/lessons/11-agents-intro.md) you built the agentic loop by hand: a `while` loop that called the LLM, executed any tool calls it returned, sent the results back, and stopped when the model produced a final answer with no more tool calls. That pattern is the foundation of every agent framework.\\n\\nIn Kestra, the `AIAgent` plugin handles that loop for you. You define the goal, the tools, and optionally a system message - Kestra drives the loop, manages conversation history, and surfaces the result as a task output.\\n\\n> Note: The flows in this lesson use `{{ secret(\'GEMINI_API_KEY\') }}`. Make sure you\'ve completed the [setup instructions](03-setup.md) to configure this secret before running them.\\n\\nThe example flows use Gemini, but the `provider` block supports any major AI provider — swap `io.kestra.plugin.ai.provider.GoogleGemini` for `OpenAI`, `Anthropic`, or others. See the [full list of supported providers](https://kestra.io/plugins/plugin-ai/provider).\\n\\nTraditional Workflow — fixed sequence, predetermined logic:\\n\\n```yaml\\ntasks:\\n  - id: step1\\n    type: Task1\\n  - id: step2\\n    type: Task2\\n  - id: step3\\n    type: Task3\\n```\\n\\nAI Agent Workflow — agent decides what to do, in what order, based on the goal:\\n\\n```yaml\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.agent.AIAgent\\n    prompt: \\"Research data engineering trends and create a report\\"\\n    tools:\\n      - WebSearch\\n      - TaskExecution\\n```\\n\\n## When to Use AI Agents\\n\\nUse AI Agents when the exact sequence of steps isn\'t known in advance, decisions depend on dynamic changing information, or you need to adapt to unexpected conditions.\\n\\nUse traditional workflows when steps are deterministic and repeatable, compliance requires exact auditable processes, or cost and latency must be minimized.\\n\\n## Anatomy of an AI Agent\\n\\n```yaml\\nid: example_agent\\nnamespace: zoomcamp\\n\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.age", "filename": "03-orchestration/lessons/06-agents.md"}, {"start": 7000, "content": "```python\\nusage = response.usage\\nusage.input_tokens, usage.output_tokens\\n```\\n\\nFor each model the provider publishes a price per million input tokens\\nand per million output tokens. Plug those numbers in to convert tokens\\nto dollars.\\n\\n```python\\ndef calculate_gpt54mini_price(input_tokens, output_tokens):\\n    INPUT_PRICE_PER_MILLION = 0.15\\n    OUTPUT_PRICE_PER_MILLION = 0.60\\n\\n    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION\\n    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION\\n    total_cost = input_cost + output_cost\\n\\n    return {\\n        \\"input_cost\\": input_cost,\\n        \\"output_cost\\": output_cost,\\n        \\"total_cost\\": total_cost,\\n    }\\n\\nresult = calculate_gpt54mini_price(652, 33)\\nprint(\\"Total cost: $\\", round(result[\\"total_cost\\"], 8))\\n```\\n\\nThis usage is only for the second API call. The first call also has\\nits own usage and its own cost. That was the call where the model\\ndecided to invoke `search`. Two calls means we pay twice. We pay even\\nmore on the second call, because we resend the full history as input.\\n\\nWith a real agent loop the model can make many calls, so the costs add\\nup. Keep an eye on `usage` while you develop.\\n\\n[← Quick RAG Revision](12-rag-revision.md) | [The Agentic Loop →](14-agentic-loop.md)", "filename": "01-agentic-rag/lessons/13-function-calling.md"}, {"start": 0, "content": "# Agents\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn Part 1 of this module we built RAG pipelines.\\n\\nEvery pipeline we wrote followed the same flow:\\n\\n- search the FAQ,\\n- build a prompt with the results,\\n- send it to the LLM.\\n\\nThis returns good answers when the user\'s query matches the documents.\\nThe search finds the right entry, the LLM reads it, and you get a\\nhelpful reply.\\n\\nOften, though, the search returns nothing useful.\\n\\n- Maybe the user made a typo.\\n- Maybe they asked the question in an unusual way.\\n- Maybe they need information from two different searches.\\n\\nWe use lexical search here, so the search looks for an exact match.\\nOne typo and it misses the entry it needed. In our pipeline there\'s\\nno recovery. The search runs once, and if it returns garbage the LLM\\ngets garbage. Our pipeline always does the same thing, no matter what.\\n\\nInstead of routing the user question straight to search, we can hand\\ncontrol to the LLM and let it drive.\\n\\nThe LLM is in charge now, and it can:\\n\\n- fix typos\\n- search again with different terms\\n- ask the user a clarifying question\\n\\nA fixed flow can\'t do any of this. Once we put the LLM in control,\\nour system becomes agentic, so it\'s flexible rather than rigid.\\n\\nAn agent uses an LLM to decide which actions to take and in which\\norder. Instead of a fixed flow, the LLM chooses what to do at each\\nstep.\\n\\nIn Part 2 of this module, we\'ll cover:\\n\\n- Function calling, so we can give the LLM tools it can use\\n- The agentic loop, where the LLM decides when to call a tool, when\\n  to call another one, and when to stop and answer\\n- Frameworks, the libraries that run this loop for you\\n\\nWe build on top of the RAG pipeline from Part 1, which used keyword\\nsearch with minsearch. If you skipped Part 1, the next lesson does a\\nquick revision and walks you through downloading the helpers.\\n\\n[← Back to module](../) | [Quick RAG Revision →](12-rag-revision.md)", "filename": "01-agentic-rag/lessons/11-agents-intro.md"}, {"start": 6000, "content": "    has_function_calls = False\\n\\n        response = openai_client.responses.create(\\n            model=model,\\n            input=messages,\\n            tools=[search_tool]\\n        )\\n\\n        messages.extend(response.output)\\n\\n        for item in response.output:\\n            if item.type == \\"function_call\\":\\n                print(\\"function_call:\\", item.name, item.arguments)\\n                call_output = make_call(item)\\n                messages.append(call_output)\\n                has_function_calls = True\\n\\n            elif item.type == \\"message\\":\\n                print(\\"ASSISTANT:\\")\\n                last_answer = item.content[0].text\\n                print(item.content[0].text)\\n\\n        it = it + 1\\n        if has_function_calls == False:\\n            break\\n\\n    return last_answer\\n```\\n\\nTry it with a question that has a typo:\\n\\n```python\\nagent_loop(instructions, \\"How do I run Olama locally?\\")\\n```\\n\\nWatch what happens. The agent searches for \\"Olama\\" and gets poor\\nresults. It then searches again with \\"Ollama\\" and finds the answer.\\nThe loop lets the model recover from a bad search on its own. That\'s\\nthe whole point of going agentic.\\n\\nAlso try the course enrollment question:\\n\\n```python\\nagent_loop(instructions, \\"I just discovered the course. Can I still join it?\\")\\n```\\n\\n## Encouraging multiple searches\\n\\nThere\'s a subtle issue here. The model often answers after the first\\nsearch, even when more searches would help. It reasons that it already\\nknows enough, so why bother. We push it to explore more by rewriting\\nthe instructions.\\n\\n```python\\ninstructions = \\"\\"\\"\\nYou\'re a course teaching assistant.\\nYou\'re given a question from a course student and your task is to answer it.\\n\\nIf you want to look up information, use the search function. \\nUse as many keywords from the user question as possible when making first requests.\\n\\nMake multiple searches. First perform search, analyze the results \\nand then perform more searches. \\n\\nAt the end, ask if there are other areas that the user wants to explore.\\n\\"\\"\\".s", "filename": "01-agentic-rag/lessons/14-agentic-loop.md"}]'}, {'role': 'tool', 'tool_call_id': 'w1msj2r1s', 'name': 'search', 'content': '[{"start": 0, "content": "# AI Agents\\n\\nVideo: [Agentic Workflows](https://youtu.be/7tvpR8EE0gs)\\n\\nIn [Module 1](../../01-agentic-rag/lessons/11-agents-intro.md) you built the agentic loop by hand: a `while` loop that called the LLM, executed any tool calls it returned, sent the results back, and stopped when the model produced a final answer with no more tool calls. That pattern is the foundation of every agent framework.\\n\\nIn Kestra, the `AIAgent` plugin handles that loop for you. You define the goal, the tools, and optionally a system message - Kestra drives the loop, manages conversation history, and surfaces the result as a task output.\\n\\n> Note: The flows in this lesson use `{{ secret(\'GEMINI_API_KEY\') }}`. Make sure you\'ve completed the [setup instructions](03-setup.md) to configure this secret before running them.\\n\\nThe example flows use Gemini, but the `provider` block supports any major AI provider — swap `io.kestra.plugin.ai.provider.GoogleGemini` for `OpenAI`, `Anthropic`, or others. See the [full list of supported providers](https://kestra.io/plugins/plugin-ai/provider).\\n\\nTraditional Workflow — fixed sequence, predetermined logic:\\n\\n```yaml\\ntasks:\\n  - id: step1\\n    type: Task1\\n  - id: step2\\n    type: Task2\\n  - id: step3\\n    type: Task3\\n```\\n\\nAI Agent Workflow — agent decides what to do, in what order, based on the goal:\\n\\n```yaml\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.agent.AIAgent\\n    prompt: \\"Research data engineering trends and create a report\\"\\n    tools:\\n      - WebSearch\\n      - TaskExecution\\n```\\n\\n## When to Use AI Agents\\n\\nUse AI Agents when the exact sequence of steps isn\'t known in advance, decisions depend on dynamic changing information, or you need to adapt to unexpected conditions.\\n\\nUse traditional workflows when steps are deterministic and repeatable, compliance requires exact auditable processes, or cost and latency must be minimized.\\n\\n## Anatomy of an AI Agent\\n\\n```yaml\\nid: example_agent\\nnamespace: zoomcamp\\n\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.age", "filename": "03-orchestration/lessons/06-agents.md"}, {"start": 1000, "content": "= 0.500 + 0.167 = 0.667\\n  B = 1/(1+1+1) + 1/(1+1+1) = 0.333 + 0.333 = 0.667\\n  C = 1/(1+2+1) + 1/(1+0+1) = 0.250 + 0.500 = 0.750\\n  D = 1/(1+3+1)              = 0.200\\n  E = 1/(1+4+1)              = 0.167\\n  F =              1/(1+2+1) = 0.250\\n  G =              1/(1+3+1) = 0.200\\n\\nFinal ranking: C, A/B (tie), F, D/G (tie), E\\n```\\n\\nC wins because it ranks high in both lists. Documents that only\\nappear in one list get lower scores.\\n\\nThe parameter `k` smooths the\\ndifferences between ranks - higher `k` means rank position matters\\nless.\\n\\nThis algorithm works with any number of ranked lists, not just two.\\nSo in our implementation we can generalize to an arbitrary number\\nof ranked results.\\n\\nLet\'s implement it:\\n\\n```python\\ndef rrf(search_results, k=1, num_results=10):\\n    scores = {}\\n    doc_map = {}\\n\\n    for results in search_results:\\n        for rank, doc in enumerate(results):\\n            key = doc[\\"question\\"]\\n            if key not in scores:\\n                scores[key] = 0\\n                doc_map[key] = doc\\n            scores[key] += 1 / (k + rank + 1)\\n\\n    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)\\n    return [doc_map[key] for key, _ in ranked[:num_results]]\\n```\\n\\nLet\'s put everything together:\\n\\n```python\\ndef hybrid_search(query, course=\\"data-engineering-zoomcamp\\", num_results=10):\\n    keyword_results = keyword_search(query, course=course, num_results=num_results)\\n    vector_results = vector_search(query, course=course, num_results=num_results)\\n    return rrf([keyword_results, vector_results], num_results=num_results)\\n```\\n\\n[← Best Practices for RAG](01-intro.md) | [Document Reranking →](03-reranking.md)", "filename": "06-best-practices/lessons/02-hybrid-search.md"}, {"start": 4000, "content": "esult.\\n\\nThe `result` is a `LoopResult` with `all_messages` (the full\\nconversation), token counts, and `cost` (computed from token usage).\\n\\n## Cost and tokens\\n\\nLook at what the call cost:\\n\\n```python\\nresult.cost\\n```\\n\\nUseful while developing - especially with multi-turn agents where one\\nprompt can trigger several model calls. The handwritten loop made you\\ncompute this by hand. The framework keeps a running total for you.\\n\\nYou can also look at the full message history.\\n\\n```python\\nresult.all_messages\\n```\\n\\nThis is just a list - the same `messages` list we maintained by hand.\\n\\n## Continuing the conversation\\n\\nTake the messages from the previous result and pass them as\\n`previous_messages` on the next `loop` call:\\n\\n```python\\nresult2 = runner.loop(\\n    prompt=\\"How do I run a different model?\\",\\n    previous_messages=result.all_messages,\\n    callback=callback,\\n)\\n```\\n\\nThe runner picks up where the last call left off, with the same agent\\nloop and an extended history. The model knows \\"different model\\" refers\\nto Ollama because it sees the previous turn in memory. Without that\\nhistory, it would have no idea what we\'re asking about.\\n\\n## Interactive chat\\n\\nFor a chat-like workflow, run the built-in input loop:\\n\\n```python\\nrunner.run()\\n```\\n\\nType questions and get answers. Type \\"stop\\" to exit.\\n\\n[← The Agentic Loop](14-agentic-loop.md) | [Other Frameworks →](16-other-frameworks.md)", "filename": "01-agentic-rag/lessons/15-frameworks.md"}, {"start": 0, "content": "# Hybrid Search\\n\\nVector search finds documents by semantic meaning, while keyword search\\nfinds documents by exact word matches. Hybrid search combines both.\\n\\nEach search produces a ranked list with scores.\\n\\nWe combine them with a weighted sum:\\n\\n```text\\nscore = alpha * vector_score + (1 - alpha) * keyword_score\\n```\\n\\nWhen `alpha = 1`, it\'s pure vector search. When `alpha = 0`, it\'s pure\\nkeyword search, and values in between give a mix.\\n\\n## Reciprocal Rank Fusion\\n\\nAnother approach is fusion: merge the ranked lists from each search\\nmethod and compute a combined score based on rankings.\\n\\nReciprocal Rank Fusion (RRF) is a simple fusion method. The score\\nfor each document is the sum of `1 / (k + rank + 1)` across all\\nlists where it appears.\\n\\nHere is how it works with an example.\\n\\n- text search: `[A, B, C, D, E]`\\n- vector search: `[C, B, F, G, A]`\\n- they have 3 documents in common (A, B, C)\\n\\nWith `k = 1`:\\n\\n```text\\nVector ranks:  C=0, B=1, F=2, G=3, A=4\\n\\nRRF scores:\\n  A = 1/(1+0+1) + 1/(1+4+1) = 0.500 + 0.167 = 0.667\\n  B = 1/(1+1+1) + 1/(1+1+1) = 0.333 + 0.333 = 0.667\\n  C = 1/(1+2+1) + 1/(1+0+1) = 0.250 + 0.500 = 0.750\\n  D = 1/(1+3+1)              = 0.200\\n  E = 1/(1+4+1)              = 0.167\\n  F =              1/(1+2+1) = 0.250\\n  G =              1/(1+3+1) = 0.200\\n\\nFinal ranking: C, A/B (tie), F, D/G (tie), E\\n```\\n\\nC wins because it ranks high in both lists. Documents that only\\nappear in one list get lower scores.\\n\\nThe parameter `k` smooths the\\ndifferences between ranks - higher `k` means rank position matters\\nless.\\n\\nThis algorithm works with any number of ranked lists, not just two.\\nSo in our implementation we can generalize to an arbitrary number\\nof ranked results.\\n\\nLet\'s implement it:\\n\\n```python\\ndef rrf(search_results, k=1, num_results=10):\\n    scores = {}\\n    doc_map = {}\\n\\n    for results in search_results:\\n        for rank, doc in enumerate(results):\\n            key = doc[\\"question\\"]\\n            if key not in scores:\\n                scores[key] = 0\\n                doc_ma", "filename": "06-best-practices/lessons/02-hybrid-search.md"}, {"start": 7000, "content": "```python\\nusage = response.usage\\nusage.input_tokens, usage.output_tokens\\n```\\n\\nFor each model the provider publishes a price per million input tokens\\nand per million output tokens. Plug those numbers in to convert tokens\\nto dollars.\\n\\n```python\\ndef calculate_gpt54mini_price(input_tokens, output_tokens):\\n    INPUT_PRICE_PER_MILLION = 0.15\\n    OUTPUT_PRICE_PER_MILLION = 0.60\\n\\n    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION\\n    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION\\n    total_cost = input_cost + output_cost\\n\\n    return {\\n        \\"input_cost\\": input_cost,\\n        \\"output_cost\\": output_cost,\\n        \\"total_cost\\": total_cost,\\n    }\\n\\nresult = calculate_gpt54mini_price(652, 33)\\nprint(\\"Total cost: $\\", round(result[\\"total_cost\\"], 8))\\n```\\n\\nThis usage is only for the second API call. The first call also has\\nits own usage and its own cost. That was the call where the model\\ndecided to invoke `search`. Two calls means we pay twice. We pay even\\nmore on the second call, because we resend the full history as input.\\n\\nWith a real agent loop the model can make many calls, so the costs add\\nup. Keep an eye on `usage` while you develop.\\n\\n[← Quick RAG Revision](12-rag-revision.md) | [The Agentic Loop →](14-agentic-loop.md)", "filename": "01-agentic-rag/lessons/13-function-calling.md"}]'}, {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'v8cex3pz6', 'function': {'arguments': '{"query":"agentic loop vs plain RAG difference"}', 'name': 'search'}, 'type': 'function'}]}, {'role': 'tool', 'tool_call_id': 'v8cex3pz6', 'name': 'search', 'content': '[{"start": 5000, "content": "unction, and serialize the result.\\n\\n```python\\nimport json\\n\\ncall = response.output[0]\\nargs = json.loads(call.arguments)\\n\\nresults = search(**args)\\nresult_json = json.dumps(results, indent=2)\\n```\\n\\nNow we send this result back to the model. First, we add the model\'s\\noutput to the conversation history - the model needs to see its own\\nfunction call. Then we add the tool result.\\n\\n```python\\nmessages.extend(response.output)\\n\\nmessages.append({\\n    \\"type\\": \\"function_call_output\\",\\n    \\"call_id\\": call.call_id,\\n    \\"output\\": result_json,\\n})\\n```\\n\\nThe `call_id` links the tool result to the specific function call the\\nmodel requested. If the model makes multiple function calls in one\\nturn, each one gets its own `call_id`.\\n\\n## Asking the model again\\n\\nWe call the API a second time with the expanded history:\\n\\n```python\\nresponse = openai_client.responses.create(\\n    model=\\"gpt-5.4-mini\\",\\n    input=messages,\\n    tools=[search_tool],\\n)\\n\\nresponse.output_text\\n```\\n\\nThis time the model has the original question, its own decision to\\ncall `search`, and the FAQ results. It can now produce a proper\\ncourse-specific answer.\\n\\nWe have to send the whole history because LLMs are stateless between\\nAPI calls. The memory is the list you send as `input`. If you send\\nonly the tool result, the model has no idea what\'s going on. So on\\nthis second call we replay everything we have so far. That means the\\nquestion, the decision to call `search`, and the result we got back.\\n\\nThat\'s the full function-calling loop for a single turn. With plain\\nRAG we made one call, and here we make two. Turning RAG agentic means\\nmore round-trips.\\n\\nPeople call this pattern \\"agentic RAG\\", \\"tool use\\", or \\"function\\ncalling\\". The idea behind all of them is the same. The LLM decides\\nwhich tools to call.\\n\\n## Token usage and cost\\n\\nWe just made two API calls instead of one. Each call we send to the\\nmodel costs money, so it\'s worth checking how much one tool-using turn\\nactually costs.\\n\\nThe response has a `usage` field with the token counts:\\n\\n", "filename": "01-agentic-rag/lessons/13-function-calling.md"}, {"start": 6000, "content": "its own decision to\\ncall `search`, and the FAQ results. It can now produce a proper\\ncourse-specific answer.\\n\\nWe have to send the whole history because LLMs are stateless between\\nAPI calls. The memory is the list you send as `input`. If you send\\nonly the tool result, the model has no idea what\'s going on. So on\\nthis second call we replay everything we have so far. That means the\\nquestion, the decision to call `search`, and the result we got back.\\n\\nThat\'s the full function-calling loop for a single turn. With plain\\nRAG we made one call, and here we make two. Turning RAG agentic means\\nmore round-trips.\\n\\nPeople call this pattern \\"agentic RAG\\", \\"tool use\\", or \\"function\\ncalling\\". The idea behind all of them is the same. The LLM decides\\nwhich tools to call.\\n\\n## Token usage and cost\\n\\nWe just made two API calls instead of one. Each call we send to the\\nmodel costs money, so it\'s worth checking how much one tool-using turn\\nactually costs.\\n\\nThe response has a `usage` field with the token counts:\\n\\n```python\\nusage = response.usage\\nusage.input_tokens, usage.output_tokens\\n```\\n\\nFor each model the provider publishes a price per million input tokens\\nand per million output tokens. Plug those numbers in to convert tokens\\nto dollars.\\n\\n```python\\ndef calculate_gpt54mini_price(input_tokens, output_tokens):\\n    INPUT_PRICE_PER_MILLION = 0.15\\n    OUTPUT_PRICE_PER_MILLION = 0.60\\n\\n    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION\\n    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION\\n    total_cost = input_cost + output_cost\\n\\n    return {\\n        \\"input_cost\\": input_cost,\\n        \\"output_cost\\": output_cost,\\n        \\"total_cost\\": total_cost,\\n    }\\n\\nresult = calculate_gpt54mini_price(652, 33)\\nprint(\\"Total cost: $\\", round(result[\\"total_cost\\"], 8))\\n```\\n\\nThis usage is only for the second API call. The first call also has\\nits own usage and its own cost. That was the call where the model\\ndecided to invoke `search`. Two calls means we pay twice. We pay even\\nmore on", "filename": "01-agentic-rag/lessons/13-function-calling.md"}, {"start": 4000, "content": "esult.\\n\\nThe `result` is a `LoopResult` with `all_messages` (the full\\nconversation), token counts, and `cost` (computed from token usage).\\n\\n## Cost and tokens\\n\\nLook at what the call cost:\\n\\n```python\\nresult.cost\\n```\\n\\nUseful while developing - especially with multi-turn agents where one\\nprompt can trigger several model calls. The handwritten loop made you\\ncompute this by hand. The framework keeps a running total for you.\\n\\nYou can also look at the full message history.\\n\\n```python\\nresult.all_messages\\n```\\n\\nThis is just a list - the same `messages` list we maintained by hand.\\n\\n## Continuing the conversation\\n\\nTake the messages from the previous result and pass them as\\n`previous_messages` on the next `loop` call:\\n\\n```python\\nresult2 = runner.loop(\\n    prompt=\\"How do I run a different model?\\",\\n    previous_messages=result.all_messages,\\n    callback=callback,\\n)\\n```\\n\\nThe runner picks up where the last call left off, with the same agent\\nloop and an extended history. The model knows \\"different model\\" refers\\nto Ollama because it sees the previous turn in memory. Without that\\nhistory, it would have no idea what we\'re asking about.\\n\\n## Interactive chat\\n\\nFor a chat-like workflow, run the built-in input loop:\\n\\n```python\\nrunner.run()\\n```\\n\\nType questions and get answers. Type \\"stop\\" to exit.\\n\\n[← The Agentic Loop](14-agentic-loop.md) | [Other Frameworks →](16-other-frameworks.md)", "filename": "01-agentic-rag/lessons/15-frameworks.md"}, {"start": 0, "content": "# AI Agents\\n\\nVideo: [Agentic Workflows](https://youtu.be/7tvpR8EE0gs)\\n\\nIn [Module 1](../../01-agentic-rag/lessons/11-agents-intro.md) you built the agentic loop by hand: a `while` loop that called the LLM, executed any tool calls it returned, sent the results back, and stopped when the model produced a final answer with no more tool calls. That pattern is the foundation of every agent framework.\\n\\nIn Kestra, the `AIAgent` plugin handles that loop for you. You define the goal, the tools, and optionally a system message - Kestra drives the loop, manages conversation history, and surfaces the result as a task output.\\n\\n> Note: The flows in this lesson use `{{ secret(\'GEMINI_API_KEY\') }}`. Make sure you\'ve completed the [setup instructions](03-setup.md) to configure this secret before running them.\\n\\nThe example flows use Gemini, but the `provider` block supports any major AI provider — swap `io.kestra.plugin.ai.provider.GoogleGemini` for `OpenAI`, `Anthropic`, or others. See the [full list of supported providers](https://kestra.io/plugins/plugin-ai/provider).\\n\\nTraditional Workflow — fixed sequence, predetermined logic:\\n\\n```yaml\\ntasks:\\n  - id: step1\\n    type: Task1\\n  - id: step2\\n    type: Task2\\n  - id: step3\\n    type: Task3\\n```\\n\\nAI Agent Workflow — agent decides what to do, in what order, based on the goal:\\n\\n```yaml\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.agent.AIAgent\\n    prompt: \\"Research data engineering trends and create a report\\"\\n    tools:\\n      - WebSearch\\n      - TaskExecution\\n```\\n\\n## When to Use AI Agents\\n\\nUse AI Agents when the exact sequence of steps isn\'t known in advance, decisions depend on dynamic changing information, or you need to adapt to unexpected conditions.\\n\\nUse traditional workflows when steps are deterministic and repeatable, compliance requires exact auditable processes, or cost and latency must be minimized.\\n\\n## Anatomy of an AI Agent\\n\\n```yaml\\nid: example_agent\\nnamespace: zoomcamp\\n\\ntasks:\\n  - id: agent\\n    type: io.kestra.plugin.ai.age", "filename": "03-orchestration/lessons/06-agents.md"}, {"start": 7000, "content": "```python\\nusage = response.usage\\nusage.input_tokens, usage.output_tokens\\n```\\n\\nFor each model the provider publishes a price per million input tokens\\nand per million output tokens. Plug those numbers in to convert tokens\\nto dollars.\\n\\n```python\\ndef calculate_gpt54mini_price(input_tokens, output_tokens):\\n    INPUT_PRICE_PER_MILLION = 0.15\\n    OUTPUT_PRICE_PER_MILLION = 0.60\\n\\n    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION\\n    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION\\n    total_cost = input_cost + output_cost\\n\\n    return {\\n        \\"input_cost\\": input_cost,\\n        \\"output_cost\\": output_cost,\\n        \\"total_cost\\": total_cost,\\n    }\\n\\nresult = calculate_gpt54mini_price(652, 33)\\nprint(\\"Total cost: $\\", round(result[\\"total_cost\\"], 8))\\n```\\n\\nThis usage is only for the second API call. The first call also has\\nits own usage and its own cost. That was the call where the model\\ndecided to invoke `search`. Two calls means we pay twice. We pay even\\nmore on the second call, because we resend the full history as input.\\n\\nWith a real agent loop the model can make many calls, so the costs add\\nup. Keep an eye on `usage` while you develop.\\n\\n[← Quick RAG Revision](12-rag-revision.md) | [The Agentic Loop →](14-agentic-loop.md)", "filename": "01-agentic-rag/lessons/13-function-calling.md"}]'}, {'role': 'assistant', 'content': "The agentic loop is a framework that allows the LLM to decide which actions to take and in which order, whereas plain RAG is a fixed flow that doesn't allow for dynamic decision-making. The agentic loop involves a while loop that calls the LLM, executes any tool calls it returns, sends the results back, and stops when the model produces a final answer with no more tool calls. This pattern is the foundation of every agent framework. In Kestra, the AIAgent plugin handles that loop for you. You define the goal, the tools, and optionally a system message - Kestra drives the loop, manages conversation history, and surfaces the result as a task output."}]
No function calls detected or final answer provided, breaking loop.

In [ ]:
# Install the toyaikit library, a small agent library for learning and experimentation.
%pip install toyaikit

In [ ]:
from toyaikit.tools import Tools

# Initialize ToyAIKit's Tools manager.
agent_tools = Tools()
# Register the 'search' function and its corresponding tool schema with agent_tools.
agent_tools.add_tool(search, search_tool)

print("agent_tools initialized and search function registered.")

In [ ]:
# Display the tool schema that ToyAIKit generated or is using for the registered tools.
agent_tools.get_tools()

In [ ]:
import json
import uuid # Added import for uuid
from toyaikit.llm import OpenAIClient, LLMClient, Message # Removed FunctionCall from import
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from typing import List, Optional, Union
from groq import Groq

from pydantic import BaseModel, Field # Added Field for FunctionCall arguments

# Import OpenAI Pydantic models for compatibility (still useful for internal parsing from Groq)
from openai.types.chat.chat_completion import ChatCompletion, Choice
from openai.types.completion_usage import CompletionUsage as OpenAICompletionUsage # Renamed to avoid conflict
from openai.types.chat.chat_completion_message import ChatCompletionMessage
from openai.types.chat.chat_completion_message_tool_call import ChatCompletionMessageToolCall, Function as OpenAI_Function # Renamed to avoid conflict

# Define ToyAIKit-compatible Pydantic models if they are not directly importable
class ToyAIKitUsage(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
    input_tokens: int = Field(alias="prompt_tokens") # Alias for toyaikit compatibility
    output_tokens: int = Field(alias="completion_tokens") # Alias for toyaikit compatibility

# Re-introducing custom FunctionCall model - This is now likely unnecessary if tool_calls is used
# class ToyAIKitFunctionCall(BaseModel):
#     name: str
#     arguments: str

# Define ToyAIKit-compatible ContentPart model
class ToyAIKitContentPart(BaseModel):
    text: str

# Assuming LLMClient and Tools are already defined in a previous cell and are in scope.

class GroqClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
        api_key: Optional[str] = None,
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqClient"
            )

        self.model = model

        if client is None:
            self.client = Groq(api_key=api_key)
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def send_request(
        self,
        chat_messages: List, # toyaikit passes this as a list of dictionaries
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_list = tools.get_tools()

        # Convert toyaikit's list of dicts to Groq-compatible message format
        groq_messages = []
        for msg in chat_messages:
            if isinstance(msg, dict):
                groq_messages.append(msg)
            elif hasattr(msg, 'model_dump'): # if it's a pydantic model
                # Handle toyaikit.llm.Message to Groq message format
                # If a toyaikit.llm.Message has tool_calls, we need to convert them
                msg_dict = msg.model_dump(exclude_unset=True)
                if 'tool_calls' in msg_dict and msg_dict['tool_calls']:
                    # Groq API expects 'tool_calls' directly in the message
                    groq_tool_calls = []
                    for tc in msg_dict['tool_calls']:
                        # Assuming tc is a dict or similar structure compatible with ChatCompletionMessageToolCall's constructor
                        # The 'function' part of the tool call needs to be an OpenAI_Function object
                        groq_tool_calls.append(ChatCompletionMessageToolCall(
                            id=tc.get('id', str(uuid.uuid4())),
                            function=OpenAI_Function(name=tc['function']['name'], arguments=tc['function']['arguments']),
                            type="function"
                        ))
                    msg_dict['tool_calls'] = groq_tool_calls
                    # Also ensure content is None for tool_call messages in Groq format
                    msg_dict['content'] = None # Groq expects None
                # Remove the custom 'type' if it's not expected by Groq for these messages
                # if 'type' in msg_dict and msg_dict['type'] != 'message':
                #     msg_dict.pop('type')
                groq_messages.append(msg_dict)
            else:
                groq_messages.append(dict(msg))

        args = dict(
            model=self.model,
            messages=groq_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )

        if 'output_format' in kwargs:
            kwargs.pop('output_format')

        args.update(kwargs)

        groq_raw_response = self.client.chat.completions.create(**args)

        # --- Start of adaptation to toyaikit.llm.Response-compatible object ---

        # 1. Adapt usage object to ToyAIKitUsage
        toyaikit_usage = None
        if hasattr(groq_raw_response, 'usage') and groq_raw_response.usage:
            groq_usage = groq_raw_response.usage
            toyaikit_usage = ToyAIKitUsage(
                prompt_tokens=groq_usage.prompt_tokens,
                completion_tokens=groq_usage.completion_tokens,
                total_tokens=groq_usage.total_tokens
            )
            # Explicitly add input_tokens and output_tokens as attributes
            # This ensures they are directly accessible, satisfying toyaikit's runner.
            setattr(toyaikit_usage, 'input_tokens', groq_usage.prompt_tokens)
            setattr(toyaikit_usage, 'output_tokens', groq_usage.completion_tokens)

        # 2. Adapt choices/messages to a list of toyaikit.llm.Message objects
        toyaikit_messages_output = []
        if groq_raw_response.choices:
            groq_choice = groq_raw_response.choices[0]
            groq_message = groq_choice.message

            # Handle tool calls first (if any)
            if groq_message.tool_calls:
                toyaikit_tool_calls_list = []
                for tc in groq_message.tool_calls: # tc is groq.types.chat.chat_completion_message_tool_call.ChatCompletionMessageToolCall
                    toyaikit_tool_calls_list.append(
                        ChatCompletionMessageToolCall( # Use the imported OpenAI type for compatibility
                            id=tc.id,
                            function=OpenAI_Function(
                                name=tc.function.name,
                                arguments=tc.function.arguments
                            ),
                            type="function" # This is the type of the tool call itself
                        )
                    )

                # Append the message with tool_calls
                toyaikit_messages_output.append(
                    Message(
                        id=str(uuid.uuid4()), # Generate a unique ID for the message
                        role="assistant", # Tool calls are initiated by the assistant
                        # For tool calls, content should be an empty list as per toyaikit.llm.Message validation.
                        # However, DisplayingRunnerCallback.on_message expects content[0].
                        # Provide a minimal content part to satisfy this, returning an empty string.
                        content=[ToyAIKitContentPart(text="")],
                        tool_calls=toyaikit_tool_calls_list, # Pass the list of tool calls
                        type="message", # Must be 'message' even for function calls as per ValidationError
                        model=self.model,
                        usage=toyaikit_usage.model_dump() if toyaikit_usage else None # Convert to dict
                    )
                )

            # Handle content message (if any content is present)
            if groq_message.content:
                toyaikit_messages_output.append(
                    Message(
                        id=str(uuid.uuid4()), # Generate a unique ID
                        role=groq_message.role,
                        content=[ToyAIKitContentPart(text=groq_message.content)], # Wrap content in ContentPart
                        type="message", # Explicitly set type to "message" for text content
                        model=self.model,
                        usage=toyaikit_usage.model_dump() if toyaikit_usage else None # Convert to dict
                    )
                )

        # 3. Construct a toyaikit-compatible response object
        # This class will mimic toyaikit.llm.OpenAIClient.Response
        class ToyAIKitCompatibleResponse(BaseModel):
            output: List[Message]
            usage: Optional[ToyAIKitUsage] = None # Use our new ToyAIKitUsage here

        return ToyAIKitCompatibleResponse(output=toyaikit_messages_output, usage=toyaikit_usage)

    @property
    def responses(self):
        """Compatibility layer for RAGBase and other tools expecting .responses.create"""
        return self

    def create(self, model: str = None, input: List = None, **kwargs):
        """Compatibility method for the 'responses' API."""
        return self.send_request(
            chat_messages=input,
            model=model or self.model,
            **kwargs
        )

# Initialize an IPythonChatInterface for interactive chat display in the notebook.
chat_interface = IPythonChatInterface()
# Initialize a DisplayingRunnerCallback to visualize model messages and tool calls.
callback = DisplayingRunnerCallback(chat_interface)

# Initialize an OpenAIClient for ToyAIKit, using the same model as in the agentic loop
# and explicitly passing the Groq API key for authentication.
llm_client_toyaikit = GroqClient(model="openai/gpt-oss-120b", api_key=api_key_groq) # Pass the api_key_groq here
# llm_client_toyaikit =
# Initialize the OpenAIResponsesRunner, which orchestrates the agentic loop.
# It takes registered tools, developer instructions, the chat interface, and the LLM client.
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client_toyaikit
)

print("Chat interface and runner initialized.")

In [ ]:
# Execute the agentic loop using the runner with the defined question and callback.
# The 'question' variable is taken from Q6's definition.
result = runner.loop(
    prompt=question, # Using the 'question' from Q6
    callback=callback,
)

In [ ]:
# Print the total number of times the 'search' function was explicitly called by the agent.
print(f"The agent called search {search_call_count} times.")

In [ ]:
# Display the complete history of messages, including system, user, assistant, function calls, and function call outputs, from the agent's run.
print(result.all_messages)

In [ ]:
print("\n--- Analyzing Tool Calls History ---")
# Iterate through all messages in the result to analyze the structure of tool calls.
for i, message in enumerate(result.all_messages):
    # Check if the message type is a function call.
    if message.type == "function_call":
        print(f"Message {i}: Function Call - Name: {message.name}, Arguments: {message.arguments}")
    # Check if the message type is a function call output.
    elif message.type == "function_call_output":
        print(f"Message {i}: Function Call Output - Call ID: {message.call_id}, Output: {message.output}")
    # Optionally, you can also print assistant messages.
    elif message.type == "message":
        # print(f"Message {i}: Assistant Message - Content: {message.content[0].text}")
        pass

# ToyAIKit

Видео: [Смотреть этот урок](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

Написанный вручную агентный цикл из предыдущего урока полезен для обучения, но он довольно однообразен. Каждый раз при создании нового агента вам пришлось бы писать один и тот же цикл `while`, одну и ту же обработку вызовов функций и управление сообщениями.

`ToyAIKit` инкапсулирует этот паттерн, позволяя вам сосредоточиться на инструментах, промптах и поведении. Мы построили его вместе на одном из воркшопов DataTalks.Club некоторое время назад. Он делает то же самое, что и наш ручной цикл, но с меньшим количеством шаблонного кода. Если вы заглянете в код `runners`, вы найдете там тот же цикл `while True`, который мы писали сами.

Я использую его здесь намеренно, потому что не хочу выбирать победителя среди промышленных фреймворков. `ToyAIKit` маленький и легко читается, поэтому, если что-то сломается, вы сможете увидеть, что именно произошло. Это делает его удобным для разработки и отладки локально перед переходом в продакшн.

Одно предостережение: `ToyAIKit` — это библиотека для обучения и экспериментов, она НЕ предназначена для использования в реальных проектах (production). Мы используем ее, потому что она минималистична и наглядна.

## Настройка

Установите библиотеку:

```bash
uv add toyaikit
```

Импортируйте необходимые классы:

```python
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
```

## Регистрация инструмента

Мы регистрируем нашу функцию `search` вместе со схемой из предыдущих уроков:

```python
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
```

## Автоматическая генерация схемы в ToyAIKit

Писать схему вручную утомительно, и мы не хотим делать это для каждой функции. На самом деле, это и не обязательно.

Если мы добавим аннотации типов и строку документации (docstring) к функции `search`, `ToyAIKit` прочитает их и сам составит схему за нас:

```python
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )
```

Затем зарегистрируйте ее без передачи схемы:

```python
agent_tools = Tools()
agent_tools.add_tool(search)
```

Вы можете посмотреть, что сгенерировал `ToyAIKit`:

```python
agent_tools.get_tools()
```

На выходе вы получите ту же JSON-схему, которую мы писали вручную в уроке про вызов функций. `ToyAIKit` создал ее на основе докстринга и аннотации типа.

Любой современный агентный фреймворк использует этот прием. Он считывает типизированную функцию Python с документацией и строит на ее основе схему. Так работают OpenAI Agents SDK, PydanticAI, LangChain и Google ADK. Вы пишете инструмент, а фреймворк сам понимает, как его описать.

## Интерфейс чата и раннер (runner)

Создайте интерфейс чата и колбэк, а затем постройте раннер:

```python
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
```

`chat_interface` отвечает за отображение в ноутбуке. `callback` отрисовывает сообщения модели и вызовы инструментов по мере их возникновения. Раннер запускает агентный цикл — тот самый `while True`, который мы писали вручную. Он отправляет сообщения, выполняет вызовы функций, добавляет результаты инструментов обратно и повторяет процесс до тех пор, пока модель не закончит.

Мы намеренно выбрали здесь `gpt-5.4-mini`. Без нее `ToyAIKit` использует более простую и быструю модель по умолчанию, которая не так надежно следует инструкциям.

## Запуск одного промпта

Запустите один запрос:

```python
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)
```

Мы специально использовали опечатку «Olama». Агент выполняет поиск, получает плохие результаты, а затем пробует еще раз с «Ollama». Механизм восстановления такой же, как в ручном цикле. Вывод в ноутбуке выглядит гораздо приятнее: каждый вызов инструмента и сообщение отображаются по очереди, так что вы можете изучить каждый результат поиска.

`result` — это объект `LoopResult`, содержащий `all_messages` (всю историю диалога), количество токенов и `cost` (стоимость, рассчитанную на основе использования токенов).

## Стоимость и токены

Посмотрите, сколько стоил этот вызов:

```python
result.cost
```

Это полезно при разработке — особенно для многоходовых агентов, где один запрос может вызвать несколько обращений к модели. В ручном цикле вам приходилось считать это самим. Фреймворк делает это за вас автоматически.

Вы также можете просмотреть полную историю сообщений:

```python
result.all_messages
```

Это просто список — тот же список `messages`, который мы вели вручную.

## Продолжение диалога

Возьмите сообщения из предыдущего результата и передайте их как `previous_messages` при следующем вызове `loop`:

```python
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)
```

Раннер продолжит с того места, где остановился предыдущий вызов, используя тот же агентный цикл и расширенную историю. Модель поймет, что под «другой моделью» подразумевается Ollama, так как она видит предыдущий ход в памяти. Без этой истории она бы понятия не имела, о чем идет речь.

## Интерактивный чат

Для работы в режиме чата запустите встроенный цикл ввода:

```python
runner.run()
```

Вводите вопросы и получайте ответы. Введите «stop», чтобы выйти.

[← Агентный цикл](14-agentic-loop.md) | [Другие фреймворки →](16-other-frameworks.md)


In [ ]:
from typing import List, Optional

from openai import OpenAI
from pydantic import BaseModel

from openai.types.chat.chat_completion import ChatCompletion
from openai.types.chat.parsed_chat_completion import ParsedChatCompletion
from openai.types.responses.response import Response
from openai.types.responses.parsed_response import ParsedResponse

from anthropic.types import Message, RawMessageStopEvent

from toyaikit.tools import Tools


class LLMClient:
    def send_request(self, chat_messages: List, tools: Tools = None):
        raise NotImplementedError("Subclasses must implement this method")

class GroqClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
        api_key: Optional[str] = None, # Added api_key parameter
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqClient"
            )

        self.model = model

        if client is None:
            # Pass the api_key directly to the Groq client
            self.client = Groq(api_key=api_key)
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def send_request(
        self,
        chat_messages: List,
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_list = tools.get_tools()

        args = dict(
            model=self.model,
            messages=chat_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )

        # Remove 'output_format' if present, as Groq API does not support it directly
        # or it is handled differently by the Groq client. ToyAIKit passes it by default.
        if 'output_format' in kwargs:
            kwargs.pop('output_format')

        args.update(kwargs)

        return self.client.chat.completions.create(**args)

    @property
    def responses(self):
        """Compatibility layer for RAGBase and other tools expecting .responses.create"""
        return self

    def create(self, model: str = None, input: List = None, **kwargs):
        """Compatibility method for the 'responses' API."""
        return self.send_request(
            chat_messages=input,
            model=model or self.model,
            **kwargs
        )

class GroqChatCompletionsClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqChatCompletionsClient"
            )

        self.model = model

        if client is None:
            self.client = Groq()
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def convert_single_tool(self, tool):
        """
        Convert a single OpenAI tool/function API dict to Groq-compatible format.
        """
        return {
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": tool["parameters"],
            },
        }

    def convert_api_tools_to_chat_functions(self, api_tools):
        """
        Convert a list of OpenAI API tools to Groq-compatible format.
        """
        chat_functions = []

        for tool in api_tools:
            converted = self.convert_single_tool(tool)
            chat_functions.append(converted)

        return chat_functions

    def send_request(
        self,
        chat_messages: List,
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_requests_format = tools.get_tools()
            tools_list = self.convert_api_tools_to_chat_functions(
                tools_requests_format,
            )

        args = dict(
            model=self.model,
            messages=chat_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )
        args.update(kwargs)

        return self.client.chat.completions.create(**args)

In [ ]:
import json
import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Callable, Generic, TypeVar

from openai.types.chat.chat_completion_function_message_param import (
    ChatCompletionFunctionMessageParam,
)
from openai.types.chat.chat_completion_system_message_param import (
    ChatCompletionSystemMessageParam,
)
from openai.types.chat.chat_completion_user_message_param import (
    ChatCompletionUserMessageParam,
)
from openai.types.responses.easy_input_message import EasyInputMessage
from openai.types.responses.response_function_tool_call import ResponseFunctionToolCall
from pydantic import BaseModel

from toyaikit.chat.interface import ChatInterface
from toyaikit.llm import LLMClient
from toyaikit.pricing import CostInfo, PricingConfig, TokenUsage
from toyaikit.tools import Tools

# T must be either a str or a (subclass)
# instance of pydantic BaseModel
T = TypeVar("T", str, BaseModel)


def _get_tool_call_output(call_result) -> str:
    """Extract output from tool call result, handling both dict and object types."""
    if isinstance(call_result, dict):
        return call_result.get("output", "")
    return getattr(call_result, "output", "")


@dataclass
class LoopResult(Generic[T]):
    new_messages: list
    all_messages: list
    tokens: TokenUsage
    cost: CostInfo | None
    last_message: T


class RunnerCallback(ABC):
    """Abstract base class for different chat runners."""

    @abstractmethod
    def on_function_call(self, function_call: dict, result: str):
        """
        Called when a function call is made.
        """
        pass

    @abstractmethod
    def on_message(self, message: dict):
        """
        Called when a message is received.
        """
        pass

    @abstractmethod
    def on_reasoning(self, reasoning: str):
        """
        Called when a reasoning is received.
        """
        pass

    @abstractmethod
    def on_response(self, response):
        pass


class ChatRunner(ABC):
    """Abstract base class for different chat runners."""

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_type: BaseModel = None
    ) -> LoopResult:
        """
        Do one tool-call loop.
        """
        pass

    @abstractmethod
    def run(self, previous_messages: list = None) -> LoopResult:
        """
        Repeast tool call loops until user asks to stop
        """
        pass


class DisplayingRunnerCallback(RunnerCallback):
    def __init__(self, chat_interface: ChatInterface):
        self.chat_interface = chat_interface

    def on_function_call(self, function_call, result):
        self.chat_interface.display_function_call(
            function_call.name,
            function_call.arguments,
            result,
        )

    def on_message(self, message):
        self.chat_interface.display_response(message)

    def on_reasoning(self, reasoning):
        self.chat_interface.display_reasoning(reasoning)

    def on_response(self, response):
        self.chat_interface.display("-> Response received")


class BaseToolUsingRunner(ChatRunner):
    """Base class for runners that use tools and LLM clients.

    Provides common initialization and run() method implementation.
    Subclasses only need to implement the loop() method.
    """

    def __init__(
        self,
        tools: Tools = None,
        developer_prompt: str = "You're a helpful assistant.",
        chat_interface: ChatInterface = None,
        llm_client: LLMClient = None,
        pricing_config: PricingConfig = None,
    ):
        self.tools = tools
        self.developer_prompt = developer_prompt
        self.chat_interface = chat_interface
        self.llm_client = llm_client
        self.displaying_callback = DisplayingRunnerCallback(chat_interface)
        self.pricing_config = pricing_config or PricingConfig()

    @abstractmethod
    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        """Execute one tool-call loop. Must be implemented by subclasses."""
        pass

    def run(
        self,
        previous_messages: list = None,
        stop_criteria: Callable = None,
    ) -> LoopResult:
        """Repeat tool-call loops until user asks to stop."""
        chat_messages = self._initialize_messages(previous_messages)

        total_input_tokens = 0
        total_output_tokens = 0
        last_message_text = ""

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            loop_result = self.loop(
                prompt=user_input,
                previous_messages=chat_messages,
                callback=self.displaying_callback,
            )

            chat_messages.extend(loop_result.new_messages)
            total_input_tokens += loop_result.tokens.input_tokens
            total_output_tokens += loop_result.tokens.output_tokens
            last_message_text = loop_result.last_message

            if stop_criteria and stop_criteria(loop_result.new_messages):
                break

        combined_cost = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )
        combined_tokens = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        return LoopResult(
            new_messages=chat_messages,
            all_messages=chat_messages,
            tokens=combined_tokens,
            cost=combined_cost,
            last_message=last_message_text,
        )

    @abstractmethod
    def _initialize_messages(self, previous_messages: list = None) -> list:
        """Initialize chat messages. Must be implemented by subclasses."""
        pass


class OpenAIResponsesRunner(BaseToolUsingRunner):
    """Runner for OpenAI responses API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [
                EasyInputMessage(
                    role="developer",
                    content=self.developer_prompt,
                )
            ]
        return list(previous_messages)  # Return a copy

    def loop(
        self,
        prompt: str,
        previous_messages: list[dict] = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append(
                EasyInputMessage(
                    role="developer",
                    content=self.developer_prompt,
                )
            )
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        chat_messages.append(
            EasyInputMessage(
                role="user",
                content=prompt,
            )
        )

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            response = self.llm_client.send_request(
                chat_messages=chat_messages,
                tools=self.tools,
                output_format=output_format,
            )
            if callback:
                callback.on_response(response)

            if hasattr(response, "usage") and response.usage:
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens

            has_function_calls = False

            chat_messages.extend(response.output)

            for entry in response.output:
                if entry.type == "function_call":
                    result = self.tools.function_call(entry)
                    chat_messages.append(result)
                    if callback:
                        callback.on_function_call(entry, result['output'])
                    has_function_calls = True

                elif entry.type == "message":
                    if callback:
                        callback.on_message(entry.content[0].text)

            if not has_function_calls:
                break

        cost_info = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        last_message_text = ""
        last_message = None
        for entry in reversed(response.output):
            if entry.type == "message":
                last_message_text = entry.content[0].text
                if output_format:
                    last_message = output_format.model_validate_json(last_message_text)
                else:
                    last_message = last_message_text
                break

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost_info,
            last_message=last_message,
        )


class OpenAIAgentsSDKRunner(ChatRunner):
    """Runner for OpenAI Agents SDK."""

    def __init__(self, chat_interface: ChatInterface, agent):
        try:
            from agents import Runner
        except ImportError:
            raise ImportError(
                "Please run 'pip install openai-agents' to use this feature"
            )

        self.agent = agent
        self.runner = Runner()
        self.chat_interface = chat_interface

    async def run(self) -> None:
        from agents import SQLiteSession

        session_id = f"chat_session_{uuid.uuid4().hex[:8]}"
        session = SQLiteSession(session_id)

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            result = await self.runner.run(
                self.agent, input=user_input, session=session
            )

            func_calls = {}
            for ni in result.new_items:
                raw = ni.raw_item
                if ni.type == "tool_call_item":
                    func_calls[raw.call_id] = raw

            for ni in result.new_items:
                raw = ni.raw_item

                if ni.type == "handoff_call_item":
                    raw = ni.raw_item
                    self.chat_interface.display(f"handoff: {raw.name}")

                if ni.type == "handoff_output_item":
                    self.chat_interface.display(
                        f"handoff: {ni.target_agent.name} -> {ni.source_agent.name} successful"
                    )

                if ni.type == "tool_call_output_item":
                    call_id = raw["call_id"]
                    if call_id not in func_calls:
                        self.chat_interface.display(
                            f"error: cannot find the call parameters for {call_id=}"
                        )
                    else:
                        func_call = func_calls[call_id]
                        self.chat_interface.display_function_call(
                            func_call.name, func_call.arguments, raw["output"]
                        )

                if ni.type == "message_output_item":
                    md = raw.content[0].text
                    self.chat_interface.display_response(md)


class PydanticAIRunner(ChatRunner):
    """Runner for Pydantic AI."""

    def __init__(self, chat_interface: ChatInterface, agent):
        self.chat_interface = chat_interface
        self.agent = agent

    async def run(self) -> None:
        message_history = []

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            result = await self.agent.run(
                user_prompt=user_input,
                message_history=message_history
            )

            messages = result.new_messages()
            tool_calls = {}

            for m in messages:
                for part in m.parts:
                    kind = part.part_kind

                    if kind == "text":
                        self.chat_interface.display_response(part.content)

                    elif kind == "tool-call":
                        tool_calls[part.tool_call_id] = part

                    elif kind == "tool-return":
                        call = tool_calls.get(part.tool_call_id)

                        raw_result = part.content
                        if raw_result is None:
                            result_str = ""
                        elif isinstance(raw_result, str):
                            result_str = raw_result
                        else:
                            result_str = json.dumps(raw_result, indent=2, default=str)

                        self.chat_interface.display_function_call(
                            call.tool_name,
                            json.dumps(call.args),
                            result_str
                        )

            message_history.extend(messages)



class OpenAIChatCompletionsRunner(BaseToolUsingRunner):
    """Runner for OpenAI chat completions API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [
                ChatCompletionSystemMessageParam(
                    role="system",
                    content=self.developer_prompt
                )
            ]
        return list(previous_messages)  # Return a copy

    @staticmethod
    def convert_function_output_to_tool_message(data):
        return ChatCompletionFunctionMessageParam(
            role="tool",
            tool_call_id=data["call_id"],
            content=data["output"],
        )

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append(
                ChatCompletionSystemMessageParam(
                    role="system",
                    content=self.developer_prompt
                )
            )
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        user_message = ChatCompletionUserMessageParam(
            role="user",
            content=prompt
        )
        chat_messages.append(user_message)

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            reponse = self.llm_client.send_request(
                chat_messages, self.tools, output_format
            )

            if callback:
                callback.on_response(reponse)

            if reponse.usage:
                total_input_tokens += reponse.usage.prompt_tokens
                total_output_tokens += reponse.usage.completion_tokens

            first_choice = reponse.choices[0]
            message_response = first_choice.message
            chat_messages.append(message_response)

            if hasattr(message_response, "reasoning_content"):
                reasoning = (message_response.reasoning_content or "").strip()
                if reasoning != "" and callback:
                    callback.on_reasoning(reasoning)

            content = (message_response.content or "").strip()
            if content != "" and callback:
                callback.on_message(content)

            calls = []

            if hasattr(message_response, "tool_calls"):
                calls = message_response.tool_calls

            if calls is None:
                break

            if len(calls) == 0:
                break

            for call in calls:
                function_call = ResponseFunctionToolCall(
                    type="function_call",
                    name=call.function.name,
                    arguments=call.function.arguments,
                    call_id=call.id,
                )

                call_result = self.tools.function_call(function_call)
                call_result = self.convert_function_output_to_tool_message(call_result)

                chat_messages.append(call_result)

                if callback:
                    content_val = getattr(call_result, "content", None)
                    if content_val is None and isinstance(call_result, dict):
                        content_val = call_result.get("content")
                    callback.on_function_call(function_call, content_val)

        cost = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        last_message_text = (message_response.content or "").strip()
        if output_format:
            last_message = output_format.model_validate_json(last_message_text)
        else:
            last_message = last_message_text

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost,
            last_message=last_message,
        )


class AnthropicMessagesRunner(BaseToolUsingRunner):
    """Runner for Anthropic Messages API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [{
                "role": "system",
                "content": self.developer_prompt
            }]
        return list(previous_messages)  # Return a copy

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append({
                "role": "system",
                "content": self.developer_prompt
            })
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        chat_messages.append({
            "role": "user",
            "content": prompt
        })

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            response = self.llm_client.send_request(
                chat_messages=chat_messages,
                tools=self.tools,
                output_format=output_format,
            )

            if callback:
                callback.on_response(response)

            # Track token usage
            if hasattr(response, "usage") and response.usage:
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens

            # Process the response
            assistant_message = {
                "role": "assistant",
                "content": response.content
            }
            chat_messages.append(assistant_message)

            has_tool_calls = False
            text_content = []

            for block in response.content:
                if block.type == "text":
                    text_content.append(block.text)
                    if callback:
                        callback.on_message(block.text)

                elif block.type == "tool_use":
                    has_tool_calls = True
                    function_call = ResponseFunctionToolCall(
                        type="function_call",
                        name=block.name,
                        arguments=json.dumps(block.input),
                        call_id=block.id,
                    )

                    call_result = self.tools.function_call(function_call)
                    result_output = _get_tool_call_output(call_result)

                    # Anthropic expects tool results in a user message with tool_result blocks
                    tool_result_message = {
                        "role": "tool",
                        "tool_call_id": block.id,
                        "content": result_output
                    }
                    chat_messages.append(tool_result_message)

                    if callback:
                        callback.on_function_call(function_call, result_output)

            if not has_tool_calls:
                break

        cost_info = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        # Extract last message text
        last_message_text = ""
        last_message = None
        if text_content:
            last_message_text = "".join(text_content)
            if output_format:
                last_message = output_format.model_validate_json(last_message_text)
            else:
                last_message = last_message_text

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost_info,
            last_message=last_message,
        )

In [ ]:
🚀 Module 1 of LLM Zoomcamp by @DataTalksClub complete!

Just finished Module 1 - Agentic RAG. Learned how to:

✅ Build a RAG system from scratch in plain Python
✅ Index and search documents with minsearch
✅ Chunk long documents for better retrieval
✅ Turn the RAG pipeline into an agent with function calling

Here's my homework solution: <LINK>

Following along with this amazing free course - who else is learning to build with LLMs?

You can sign up here: https://github.com/DataTalksClub/llm-zoomcamp/